# register_overlay
Registers **AbPAS (brightfield RGB)** and **DAPI/Hoechst (fluorescence)** CZI
image pairs acquired on different microscope cameras, then saves a 4-channel
OME-TIFF and a composite PNG preview.

**Pipeline (5 steps)**
1. Read CZI files — extract pixel sizes & stage coordinates
2. Resample DAPI to AbPAS pixel size (≈ ×2 upscale)
3. Crop resampled DAPI to AbPAS field of view via stage coordinates
4. Refine alignment with phase cross-correlation
5. Save 4-channel OME-TIFF + composite PNG preview


In [ ]:
import os, re, sys, glob, warnings

import numpy as np
import tifffile
from biochemical_stain_assessment.czi_io import read_czi
from scipy.ndimage import zoom as ndzoom
from skimage.registration import phase_cross_correlation
from skimage.color import rgb2gray
from skimage.transform import resize as sk_resize

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings("ignore")

print("Imports OK ✓")


: 

## Configuration

In [ ]:
# ── Edit these two paths ─────────────────────────────────────────────────────
DATA_DIR = os.path.dirname(os.path.abspath("register_overlay.ipynb"))
OUT_DIR  = os.path.join(DATA_DIR, "registered_output")

# Processing options
REFINE          = True   # run phase cross-correlation refinement
MAX_SHIFT_PX    = 50     # reject cross-corr shifts larger than this (pixels)
DOWNSAMPLE_CC   = 4      # downsampling factor for cross-correlation
DS_DISPLAY      = 32     # downsampling factor for notebook visualisations

print(f"Scanning : {DATA_DIR}")
print(f"Output   : {OUT_DIR}")
print(f"Refine   : {REFINE}  (max shift = {MAX_SHIFT_PX} px, CC ds = {DOWNSAMPLE_CC}×)")


## Helper functions

In [ ]:
def resample_to_pixel_size(img, src_px, tgt_px):
    factor = src_px / tgt_px
    if abs(factor - 1.0) < 1e-4:
        return img
    src_dtype = img.dtype
    img_f = img.astype(np.float32)
    out = ndzoom(img_f, factor, order=1) if img_f.ndim == 2           else ndzoom(img_f, (factor, factor, 1), order=1)
    info = np.iinfo(src_dtype) if np.issubdtype(src_dtype, np.integer)            else np.finfo(src_dtype)
    return np.clip(out, info.min, info.max).astype(src_dtype)


def crop_to_abpas_fov(dapi_rs, h_abpas, w_abpas,
                      cx_abpas, cy_abpas, cx_dapi, cy_dapi, tgt_px):
    h_dr, w_dr = dapi_rs.shape[:2]
    dx_px =  (cx_abpas - cx_dapi) / tgt_px
    dy_px = -(cy_abpas - cy_dapi) / tgt_px
    abpas_cy = h_dr / 2.0 + dy_px
    abpas_cx = w_dr / 2.0 + dx_px
    r0 = int(round(abpas_cy - h_abpas / 2.0))
    r1 = r0 + h_abpas
    c0 = int(round(abpas_cx - w_abpas / 2.0))
    c1 = c0 + w_abpas
    r0 = max(0, r0);  r1 = min(h_dr, r1)
    c0 = max(0, c0);  c1 = min(w_dr, c1)
    return dapi_rs[r0:r1, c0:c1], r0, c0


def refine_registration(abpas_gray, dapi_cropped,
                        max_shift_px=MAX_SHIFT_PX,
                        downsample_factor=DOWNSAMPLE_CC):
    f = downsample_factor
    target_shape = (abpas_gray.shape[0] // f, abpas_gray.shape[1] // f)
    ref = sk_resize(abpas_gray.astype(np.float32), target_shape, anti_aliasing=True)
    mov = sk_resize(dapi_cropped.astype(np.float32), target_shape, anti_aliasing=True)
    ref = (ref - ref.min()) / (ref.max() - ref.min() + 1e-9)
    mov = (mov - mov.min()) / (mov.max() - mov.min() + 1e-9)
    ref_inv = 1.0 - ref
    shift_ds, _, _ = phase_cross_correlation(ref_inv, mov, upsample_factor=4,
                                             normalization=None)
    shift_full = (float(shift_ds[0]) * f, float(shift_ds[1]) * f)
    if abs(shift_full[0]) > max_shift_px or abs(shift_full[1]) > max_shift_px:
        print(f"    ⚠  Shift {shift_full} > {max_shift_px} px — keeping stage coords.")
        return (0.0, 0.0), ref_inv, mov
    return shift_full, ref_inv, mov


def apply_shift_and_crop(arr, shift_row, shift_col):
    dr = int(round(shift_row));  dc = int(round(shift_col))
    h, w = arr.shape[:2]
    r0 = max(0, dr);  r1 = min(h, h + dr)
    c0 = max(0, dc);  c1 = min(w, w + dc)
    cropped = arr[r0:r1, c0:c1]
    if cropped.shape[:2] != (h, w):
        pad = [(max(0,-dr), max(0,dr)), (max(0,-dc), max(0,dc))]
        if arr.ndim == 3:
            pad.append((0, 0))
        cropped = np.pad(cropped, pad, mode='constant')
    return cropped[:h, :w]


def save_ome_tiff(path, abpas_rgb, dapi, px_um, sample_id):
    r16 = abpas_rgb[:,:,0].astype(np.uint16) * 257
    g16 = abpas_rgb[:,:,1].astype(np.uint16) * 257
    b16 = abpas_rgb[:,:,2].astype(np.uint16) * 257
    d   = dapi.astype(np.float32)
    d_n = ((d - d.min()) / (d.max() - d.min() + 1e-9) * 65535).astype(np.uint16)
    stack = np.stack([r16, g16, b16, d_n], axis=0)
    metadata = {
        'axes': 'CYX',
        'PhysicalSizeX': px_um, 'PhysicalSizeXUnit': 'µm',
        'PhysicalSizeY': px_um, 'PhysicalSizeYUnit': 'µm',
        'Channel': {'Name': ['AbPAS_R','AbPAS_G','AbPAS_B','DAPI'],
                    'Color': [0xFF0000FF, 0x00FF00FF, 0x0000FFFF, 0x00FFFFFF]},
    }
    tifffile.imwrite(path, stack, photometric='minisblack', metadata=metadata,
                     compression='deflate', compressionargs={'level':6},
                     imagej=False, ome=True)
    print(f"  → OME-TIFF : {os.path.basename(path)}")


def save_preview_png(path, abpas_rgb, dapi, downsample=4):
    from PIL import Image
    f = 1.0 / downsample
    h, w = abpas_rgb.shape[:2]
    sh, sw = int(h*f), int(w*f)
    ab = sk_resize(abpas_rgb.astype(np.float32)/255.0, (sh,sw), anti_aliasing=True)
    d  = dapi.astype(np.float32)
    d  = (d - d.min()) / (d.max() - d.min() + 1e-9)
    ds = sk_resize(d, (sh,sw), anti_aliasing=True)
    alpha = 0.40
    comp = ab.copy()
    comp[:,:,0] = np.clip(comp[:,:,0]*(1-alpha),       0, 1)
    comp[:,:,1] = np.clip(comp[:,:,1] + alpha*ds,      0, 1)
    comp[:,:,2] = np.clip(comp[:,:,2] + alpha*ds,      0, 1)
    Image.fromarray((comp*255).clip(0,255).astype(np.uint8)).save(path, optimize=True)
    print(f"  → Preview  : {os.path.basename(path)}")


def find_pairs(data_dir):
    abpas_files = glob.glob(
        os.path.join(data_dir,'**','AbPAS','*.czi'), recursive=True)
    pairs = []
    for ap in sorted(abpas_files):
        m = re.search(r'_AbPAS_(.+?)\.czi$', os.path.basename(ap))
        if not m:
            continue
        sid    = m.group(1)
        parent = os.path.dirname(os.path.dirname(ap))
        hits   = glob.glob(os.path.join(parent,'DAPI',f'*_{sid}.czi'))
        if hits:
            pairs.append((sid, ap, hits[0]))
        else:
            print(f"  ⚠  No DAPI match for '{sid}' — skipping.")
    return pairs

print("Helper functions defined ✓")


## Step 1 — Discover AbPAS / DAPI pairs

In [ ]:
pairs = find_pairs(DATA_DIR)
print(f"Found {len(pairs)} pair(s):\n")
for i, (sid, ap, dp) in enumerate(pairs):
    print(f"  [{i}] {sid}")
    print(f"       AbPAS : {os.path.basename(ap)}")
    print(f"       DAPI  : {os.path.basename(dp)}")
    print()


## Step 2 — Read one pair and inspect metadata
Set `SAMPLE_IDX` to inspect a different pair.


In [ ]:
SAMPLE_IDX = 0

sample_id, abpas_path, dapi_path = pairs[SAMPLE_IDX]
print(f"Sample : {sample_id}")

print("\n[1/5] Reading CZI files …")
abpas, px_abpas, _, cx_abpas, cy_abpas, _ = read_czi(abpas_path)
dapi,  px_dapi,  _, cx_dapi,  cy_dapi,  _ = read_czi(dapi_path)

h_abpas, w_abpas = abpas.shape[:2]

print()
print("─── AbPAS ──────────────────────────────────────────────────")
print(f"  File   : {os.path.basename(abpas_path)}")
print(f"  Shape  : {abpas.shape}  dtype={abpas.dtype}")
print(f"  Pixel  : {px_abpas:.4f} µm/px")
print(f"  FOV    : {w_abpas*px_abpas/1000:.2f} mm × {h_abpas*px_abpas/1000:.2f} mm")
print(f"  Centre : ({cx_abpas:.1f}, {cy_abpas:.1f}) µm")

print()
print("─── DAPI ───────────────────────────────────────────────────")
print(f"  File   : {os.path.basename(dapi_path)}")
print(f"  Shape  : {dapi.shape}  dtype={dapi.dtype}")
print(f"  Pixel  : {px_dapi:.4f} µm/px")
print(f"  FOV    : {dapi.shape[1]*px_dapi/1000:.2f} mm × "
                f"{dapi.shape[0]*px_dapi/1000:.2f} mm")
print(f"  Centre : ({cx_dapi:.1f}, {cy_dapi:.1f}) µm")

scale = px_dapi / px_abpas
print(f"\n  Pixel size ratio (DAPI/AbPAS) : {scale:.4f}")
print(f"  Stage offset ΔX : {cx_abpas-cx_dapi:+.1f} µm  "
      f"ΔY : {cy_abpas-cy_dapi:+.1f} µm")


### Visualise raw images

In [ ]:
DS = DS_DISPLAY

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(abpas[::DS, ::DS])
axes[0].set_title(f"AbPAS (raw)\n{abpas.shape}  {px_abpas:.4f} µm/px", fontsize=11)
axes[0].axis('off')

im1 = axes[1].imshow(dapi[::DS, ::DS], cmap='magma')
axes[1].set_title(f"DAPI (raw)\n{dapi.shape}  {px_dapi:.4f} µm/px", fontsize=11)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, label='intensity (raw counts)')

fig.suptitle(f"Sample: {sample_id} — raw input images", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 3 — Resample DAPI to AbPAS pixel size

In [ ]:
print(f"[2/5] Resampling DAPI: {px_dapi:.4f} → {px_abpas:.4f} µm/px  (×{scale:.3f}) …")
dapi_rs = resample_to_pixel_size(dapi, px_dapi, px_abpas)

print(f"  Original DAPI  : {dapi.shape}")
print(f"  Resampled DAPI : {dapi_rs.shape}")
print(f"  AbPAS canvas   : ({h_abpas}, {w_abpas})")

h_dr, w_dr = dapi_rs.shape
print(f"\n  Resampled DAPI is {w_dr-w_abpas} cols and "
      f"{h_dr-h_abpas} rows larger than AbPAS canvas")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(dapi[::DS, ::DS], cmap='magma')
axes[0].set_title(f"DAPI original\n{dapi.shape}  {px_dapi:.4f} µm/px", fontsize=11)
axes[0].axis('off')

axes[1].imshow(dapi_rs[::DS, ::DS], cmap='magma')
axes[1].set_title(f"DAPI resampled\n{dapi_rs.shape}  {px_abpas:.4f} µm/px", fontsize=11)
axes[1].axis('off')

fig.suptitle(f"[2/5] Resampling — DAPI pixel size matched to AbPAS",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 4 — Crop DAPI to AbPAS field of view (stage coordinates)

In [ ]:
print("[3/5] Cropping DAPI to AbPAS FOV using stage coordinates …")
dapi_crop, r0, c0 = crop_to_abpas_fov(
    dapi_rs, h_abpas, w_abpas,
    cx_abpas, cy_abpas, cx_dapi, cy_dapi, px_abpas)

dx_px =  (cx_abpas - cx_dapi) / px_abpas
dy_px = -(cy_abpas - cy_dapi) / px_abpas

print(f"  Stage offset  →  ΔX = {dx_px:+.1f} px  ΔY = {dy_px:+.1f} px")
print(f"  Crop origin in resampled DAPI : row={r0}, col={c0}")
print(f"  Cropped DAPI shape            : {dapi_crop.shape}")
print(f"  Target AbPAS shape            : ({h_abpas}, {w_abpas})")

# Pad if needed (edge rounding)
if dapi_crop.shape[0] != h_abpas or dapi_crop.shape[1] != w_abpas:
    pad_r = h_abpas - dapi_crop.shape[0]
    pad_c = w_abpas - dapi_crop.shape[1]
    dapi_crop = np.pad(dapi_crop,
                       [(pad_r//2, pad_r-pad_r//2),
                        (pad_c//2, pad_c-pad_c//2)], mode='constant')
    print(f"  Padded to                     : {dapi_crop.shape}")

# ── Visualise the crop window in the resampled DAPI ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dapi_rs_t = dapi_rs[::DS, ::DS]
axes[0].imshow(dapi_rs_t, cmap='magma')
rect = mpatches.Rectangle(
    (c0/DS, r0/DS), w_abpas/DS, h_abpas/DS,
    linewidth=2, edgecolor='cyan', facecolor='none', label='AbPAS FOV')
axes[0].add_patch(rect)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_title("Resampled DAPI\n(cyan box = AbPAS FOV)", fontsize=11)
axes[0].axis('off')

axes[1].imshow(dapi_crop[::DS, ::DS], cmap='magma')
axes[1].set_title(f"DAPI cropped to AbPAS FOV\n{dapi_crop.shape}", fontsize=11)
axes[1].axis('off')

axes[2].imshow(abpas[::DS, ::DS])
axes[2].set_title(f"AbPAS\n{abpas.shape}", fontsize=11)
axes[2].axis('off')

fig.suptitle("[3/5] Coordinate-based crop", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 5 — Refine with phase cross-correlation

In [ ]:
if REFINE:
    print("[4/5] Phase cross-correlation refinement …")
    abpas_gray = rgb2gray(abpas.astype(np.float32) / 255.0)
    shift, ref_inv_ds, dapi_ds = refine_registration(abpas_gray, dapi_crop.astype(np.float32))
    print(f"  Residual shift : row={shift[0]:+.1f} px  col={shift[1]:+.1f} px")
    print(f"  Physical shift : row={shift[0]*px_abpas:+.2f} µm  "
          f"col={shift[1]*px_abpas:+.2f} µm")

    if shift != (0.0, 0.0):
        dapi_crop_refined = apply_shift_and_crop(dapi_crop, shift[0], shift[1])
        print(f"  Shift applied ✓")
    else:
        dapi_crop_refined = dapi_crop
        print("  No shift applied (below threshold or rejected)")

    # ── Visualise correlation inputs and detected shift ───────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(ref_inv_ds, cmap='gray')
    axes[0].set_title("AbPAS inverted (CC reference)\n(nuclei now bright)", fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(dapi_ds, cmap='magma')
    axes[1].set_title("DAPI downsampled (CC moving)", fontsize=11)
    axes[1].axis('off')

    # Overlay to show before/after
    dapi_bef = dapi_crop[::DS, ::DS].astype(np.float32)
    dapi_bef = (dapi_bef - dapi_bef.min()) / (dapi_bef.max() - dapi_bef.min() + 1e-9)
    dapi_aft = dapi_crop_refined[::DS, ::DS].astype(np.float32)
    dapi_aft = (dapi_aft - dapi_aft.min()) / (dapi_aft.max() - dapi_aft.min() + 1e-9)
    ab_g = abpas_gray[::DS, ::DS]
    ab_g = (ab_g - ab_g.min()) / (ab_g.max() - ab_g.min() + 1e-9)

    diff_rgb = np.stack([ab_g, dapi_aft, dapi_bef], axis=-1)
    axes[2].imshow(diff_rgb)
    axes[2].set_title(f"Shift visualisation\nAbPAS=red, before=blue, after=green\n"
                      f"shift=({shift[0]:+.1f}, {shift[1]:+.1f}) px", fontsize=10)
    axes[2].axis('off')

    fig.suptitle("[4/5] Phase cross-correlation refinement",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

else:
    dapi_crop_refined = dapi_crop
    print("[4/5] Skipping cross-correlation refinement (REFINE=False)")


## Step 6 — Final overlay preview

In [ ]:
def overlay_preview(abpas_rgb, dapi_arr, ds=DS_DISPLAY, alpha=0.4):
    ab = abpas_rgb[::ds, ::ds].astype(np.float32) / 255.0
    d  = dapi_arr[::ds, ::ds].astype(np.float32)
    d  = (d - d.min()) / (d.max() - d.min() + 1e-9)
    comp = ab.copy()
    comp[:,:,0] = np.clip(comp[:,:,0] - alpha*d, 0, 1)
    comp[:,:,1] = np.clip(comp[:,:,1] + alpha*d, 0, 1)
    comp[:,:,2] = np.clip(comp[:,:,2] + alpha*d, 0, 1)
    return comp

overlay_coord  = overlay_preview(abpas, dapi_crop)
overlay_final  = overlay_preview(abpas, dapi_crop_refined)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(abpas[::DS_DISPLAY, ::DS_DISPLAY])
axes[0].set_title("AbPAS (brightfield)", fontsize=11)
axes[0].axis('off')

axes[1].imshow(overlay_coord)
axes[1].set_title("After coordinate crop\n(before CC refinement)", fontsize=11)
axes[1].axis('off')

axes[2].imshow(overlay_final)
axes[2].set_title("After CC refinement\n(DAPI in cyan, α=0.4)", fontsize=11)
axes[2].axis('off')

cyan_patch = mpatches.Patch(color='cyan', label='DAPI / Hoechst')
gray_patch = mpatches.Patch(color='gray', label='AbPAS brightfield')
axes[2].legend(handles=[cyan_patch, gray_patch], loc='lower right', fontsize=9)

fig.suptitle(f"Sample: {sample_id} — final registered overlay",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 7 — Save outputs for this pair

In [ ]:
print(f"[5/5] Saving outputs for sample: {sample_id} …")
os.makedirs(OUT_DIR, exist_ok=True)

tiff_path = os.path.join(OUT_DIR, f"{sample_id}_registered.ome.tif")
png_path  = os.path.join(OUT_DIR, f"{sample_id}_preview.png")

save_ome_tiff(tiff_path, abpas, dapi_crop_refined, px_abpas, sample_id)
save_preview_png(png_path, abpas, dapi_crop_refined)

print(f"\nOutputs saved to: {OUT_DIR}")

# Show the saved preview inline
from PIL import Image as PILImage
preview_img = PILImage.open(png_path)
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(np.array(preview_img))
ax.set_title(f"{sample_id}_preview.png  ({preview_img.size[0]}×{preview_img.size[1]} px)",
             fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()


## Step 8 — Batch process all pairs
Run the full pipeline on every discovered pair.


In [ ]:
def process_pair_batch(sample_id, abpas_path, dapi_path, out_dir, refine=True):
    print(f"\n{'─'*60}")
    print(f"  Sample : {sample_id}")

    abpas, px_abpas, _, cx_abpas, cy_abpas, _ = read_czi(abpas_path)
    dapi,  px_dapi,  _, cx_dapi,  cy_dapi,  _ = read_czi(dapi_path)
    h_abpas, w_abpas = abpas.shape[:2]

    print(f"  [1/5] Read   AbPAS {abpas.shape}  px={px_abpas:.4f}")
    print(f"               DAPI  {dapi.shape}   px={px_dapi:.4f}")

    scale = px_dapi / px_abpas
    print(f"  [2/5] Resample DAPI ×{scale:.3f} …")
    dapi_rs = resample_to_pixel_size(dapi, px_dapi, px_abpas)

    print(f"  [3/5] Crop to AbPAS FOV …")
    dapi_crop, r0, c0 = crop_to_abpas_fov(
        dapi_rs, h_abpas, w_abpas,
        cx_abpas, cy_abpas, cx_dapi, cy_dapi, px_abpas)
    if dapi_crop.shape[0] != h_abpas or dapi_crop.shape[1] != w_abpas:
        pr = h_abpas - dapi_crop.shape[0]; pc = w_abpas - dapi_crop.shape[1]
        dapi_crop = np.pad(dapi_crop,
                           [(pr//2, pr-pr//2), (pc//2, pc-pc//2)], mode='constant')

    if refine:
        print(f"  [4/5] Cross-correlation refinement …")
        abpas_gray = rgb2gray(abpas.astype(np.float32) / 255.0)
        shift, _, _ = refine_registration(abpas_gray, dapi_crop.astype(np.float32))
        print(f"         shift=({shift[0]:+.1f}, {shift[1]:+.1f}) px")
        if shift != (0.0, 0.0):
            dapi_crop = apply_shift_and_crop(dapi_crop, shift[0], shift[1])
    else:
        print(f"  [4/5] Cross-correlation skipped.")

    print(f"  [5/5] Saving …")
    os.makedirs(out_dir, exist_ok=True)
    save_ome_tiff(os.path.join(out_dir, f"{sample_id}_registered.ome.tif"),
                  abpas, dapi_crop, px_abpas, sample_id)
    save_preview_png(os.path.join(out_dir, f"{sample_id}_preview.png"),
                     abpas, dapi_crop)


print(f"Processing {len(pairs)} pair(s) → {OUT_DIR}")
for sid, ap, dp in pairs:
    process_pair_batch(sid, ap, dp, OUT_DIR, refine=REFINE)

print(f"\n{'═'*60}")
print(f"Done.  Results in: {OUT_DIR}")
print("Open .ome.tif in Fiji → Bio-Formats importer")
print("4 channels: AbPAS_R, AbPAS_G, AbPAS_B, DAPI")
